In [1]:
import requests
import json
import time

In [2]:
BASE_DOMAIN_NAME = "https://ttb.utoronto.ca/"
GET_PAGEABLE_COURSES = "https://api.easi.utoronto.ca/ttb/getPageableCourses"
TEST_PATH = "http://localhost:3000"

In [3]:
class GetPageableCoursesRequest:
    courseCodeAndTitleProps: dict[str, str]
    departmentProps: list
    campuses: list
    sessions: list[str]
    requirementProps: list
    instructor: str
    courseLevels: list 
    deliveryModes: list
    dayPreferences: list
    timePreferences: list
    divisions: list[str]  # of "divisions":["APSC","ARTSC","FIS","FPEH","MUSIC","ARCLA","ERIN","SCAR"]
    creditWeights: str
    availableSpace: bool
    waitListable: bool 
    page: 1
    pageSize: 20
    direction: str # "asc" for ascending sorting presumeably
    def __init__(self):
        pass
    def validate(self):
        pass
    def __str__(self):
        pass

### Strcturing the calls

In [25]:
with open("../../data/courses.json", "r") as f:
    course_data = json.load(f)

In [26]:
course_codes = [c["course_code"] for c in course_data]

In [7]:
assert {len(c) for c in course_codes} == {8} # they should all be length 8, otherwise below code breaks

In [17]:
# create a tree such that we use minimal requests to ttb
counter = 0
def create_tree(strlist, maxl, depth) -> dict | list:
    global counter
    if depth == 0:
        counter = 1
    else:
        counter += 1
    assert depth < 8  # was designed for lenght 8 course codes
    assert len({s[:depth] for s in strlist}) == 1 # they all have the same prefix
    if len(strlist) <= maxl:
        return strlist
    elif len(strlist) > maxl:
        keys = {s[depth] for s in strlist}
        d = dict()
        for key in keys:
            l = [s for s in strlist if s[depth] == key]
            if len(l) > maxl:
                l = create_tree(l, maxl, depth+1)
            d[key] = l
        return d

In [66]:
create_tree(course_codes, 50, 0)
print(counter)

109


In [67]:
def get_optimal_queries(prefix, tree) -> list:
    l = []
    for key in tree:
        if isinstance(tree[key], list):
            l.append(prefix+key)
        elif isinstance(tree[key], dict):
            l.extend(get_optimal_queries(prefix+key, tree[key]))
    return l

In [72]:
tree = create_tree(course_codes, 100, 0)
queries = get_optimal_queries("", tree)

In [73]:
len(queries)

163

In [56]:
import time

## Sessions...

20269
20271
20269-20271

Hello

In [108]:
def getPageableCourses(query: str, divisions=["ARTSC"], page: int = 1):
    request_data = {
        "courseCodeAndTitleProps":{
            # "courseCode": f"{query}",
            "courseCode": "",
            "courseSectionCode": "",
            "courseTitle": f"{query}",
            "searchCourseDescription": True
        },
        "departmentProps":[],
        "campuses":[],
        # "sessions":["20265F","20265S","20265"],
        # "sessions": [
        #     "20265",
        #     "20265S",
        #     "20265F",
        # ],
        "sessions":["20265F","20265S","20265","20269","20271","20269-20271"], # 2026 summer + next fall
        # "sessions": [f"202{y}{i}{f}" for y in ["5", "6", "7"] for i in range(0,10) for f in ["F", "S", ""]],
        # "sessions":["20259","20261","20259-20261"],
        "requirementProps":[],
        "instructor":"",
        "courseLevels":[],
        "deliveryModes":[],
        "dayPreferences":[],
        "timePreferences":[],
        # "divisions":["ARTSC","ERIN"],
        "divisions": divisions,
        "creditWeights":[],
        "availableSpace": False,
        "waitListable": False,
        "page": page,
        "pageSize":50,
        "direction":"asc"
    }
    response = requests.post(
        url=f"{GET_PAGEABLE_COURSES}",
        json=request_data,
        headers={
            "Accept": "application/json, text/plain, */*"
        }
    )
    assert response.status_code == 200
    j = response.json()
    assert list(j.keys()) == ["payload", "status"] # has two keys, status and payload
    payload = j["payload"]
    with open(f"ttb_scrapes/{query}_{time.time_ns()}.json", "w") as f:
        json.dump(payload, f)
    return payload

In [92]:
payload = getPageableCourses("")

In [100]:
total = payload["pageableCourse"]['total']
pageSize = payload["pageableCourse"]['pageSize']

In [102]:
import math

In [106]:
pages = math.ceil(total / pageSize)

In [ ]:
import time

In [111]:
payload = getPageableCourses("", page=222)

In [110]:
for page in range(16, pages):
    payload = getPageableCourses("", page=page)
    time.sleep(3)
    print(page, len(payload["pageableCourse"]['courses']))

16 20
17 20
18 20
19 20
20 20
21 20
22 20
23 20
24 20
25 20
26 20
27 20
28 20
29 20
30 20
31 20
32 20
33 20
34 20
35 20
36 20
37 20
38 20
39 20
40 20
41 20
42 20
43 20
44 20
45 20
46 20
47 20
48 20
49 20
50 20
51 20
52 20
53 20
54 20
55 20
56 20
57 20
58 20
59 20
60 20
61 20
62 20
63 20
64 20
65 20
66 20
67 20
68 20
69 20
70 20
71 20
72 20
73 20
74 20
75 20
76 20
77 20
78 20
79 20
80 20
81 20
82 20
83 20
84 20
85 20
86 20
87 20
88 20
89 20
90 20
91 20
92 20
93 20
94 20
95 20
96 20
97 20
98 20
99 20
100 20
101 20
102 20
103 20
104 20
105 20
106 20
107 20
108 20
109 20
110 20
111 20
112 20
113 20
114 20
115 20
116 20
117 20
118 20
119 20
120 20
121 20
122 20
123 20
124 20
125 20
126 20
127 20
128 20
129 20
130 20
131 20
132 20
133 20
134 20
135 20
136 20
137 20
138 20
139 20
140 20
141 20
142 20
143 20
144 20
145 20
146 20
147 20
148 20
149 20
150 20
151 20
152 20
153 20
154 20
155 20
156 20
157 20
158 20
159 20
160 20
161 20
162 20
163 20
164 20
165 20
166 20
167 20
168 20
169 20
170 20

## Parsing and Formatting

In [5]:
import os

In [7]:
filelist = os.listdir('ttb_scrapes')

In [15]:
with open(f"ttb_scrapes/{filelist[0]}", 'r') as f:
    data = json.load(f)

In [21]:
mentioned_courses = []
for i, file in enumerate(filelist):
    print(file, i)
    with open(f"ttb_scrapes/{file}", 'r') as f:
        content = f.read()
        if len(content) == 0:
            print("Warning")
        else:
            data = json.loads(content)
            mc = [c["code"] for c in data["pageableCourse"]["courses"]]
            mentioned_courses.extend(mc)

_1777821716256203000.json 0
_1777822389016864000.json 1
_1777822078979462000.json 2
_1777821566259711000.json 3
_1777821484015567000.json 4
_1777821867172074000.json 5
_1777821841055303000.json 6
_1777821761990317000.json 7
_1777821860654751000.json 8
_1777821510012073000.json 9
_1777821824737624000.json 10
_1777821455230901000.json 11
_1777821752217525000.json 12
_1777821755475245000.json 13
_1777822157983392000.json 14
_1777821453830849000.json 15
_1777820625174742000.json 16
_1777821922968443000.json 17
_1777821814991893000.json 18
_1777821735837601000.json 19
_1777821513304675000.json 20
_1777821559699079000.json 21
_1777821880245521000.json 22
_1777821562980930000.json 23
_1777821670559920000.json 24
_1777821487245913000.json 25
_1777821745664127000.json 26
_1777821992491066000.json 27
_1777822098512977000.json 28
_1777821686939354000.json 29
_1777821650996763000.json 30
_1777821989201543000.json 31
_1777821742394469000.json 32
_1777821503476304000.json 33
_1777821532849479000.jso

In [13]:
data["pageableCourse"]["courses"][0]

{'id': '696f7d2320609159511a66b2',
 'name': 'Advanced Geological Field Methods',
 'ucName': None,
 'code': 'ESS324H0',
 'sectionCode': 'F',
 'campus': 'St. George',
 'sessions': ['20265F'],
 'sections': [{'name': 'LEC3001',
   'type': 'Lecture',
   'teachMethod': 'LEC',
   'sectionNumber': '3001',
   'meetingTimes': [],
   'firstMeeting': None,
   'instructors': [{'firstName': 'Daniel', 'lastName': 'Gregory'}],
   'currentEnrolment': 9,
   'maxEnrolment': 9999,
   'subTitle': '',
   'cancelInd': 'N',
   'waitlistInd': 'N',
   'deliveryModes': [{'session': '20265F', 'mode': 'INPER'}],
   'currentWaitlist': 0,
   'enrolmentInd': 'E',
   'tbaInd': 'N',
   'openLimitInd': 'N',
   'notes': [{'name': 'Section Note', 'type': 'SECTION', 'content': ''}],
   'enrolmentControls': [{'yearOfStudy': '*',
     'post': {'code': '*', 'name': 'All'},
     'subject': {'code': '*', 'name': 'All'},
     'subjectPost': {'code': '*', 'name': 'All'},
     'typeOfProgram': {'code': '*', 'name': 'All'},
     'd

In [28]:
mc = set(mentioned_courses)
cc = set(course_codes)
print(len(mc), len(cc))

3322 5344


In [ ]:
with open("../../data/offered_courses.txt", "w") as f:
    for c in mc:
        f.write(c)
        f.write('\n')

In [35]:
mc.difference(cc)

{'AMS201H1',
 'AMS410H1',
 'ANT339H1',
 'ANT359H1',
 'ANT453H1',
 'ARH306Y0',
 'ARH361H0',
 'ARH361Y0',
 'BMS310Y1',
 'BMS390H1',
 'BMS422H1',
 'CAR130H1',
 'CAR391H1',
 'CAR490Y1',
 'CAR491H1',
 'CAS101H1',
 'CAS203H1',
 'CHM229H1',
 'CHM399Y1',
 'CIN198H1',
 'CJS195H1',
 'CJS196H1',
 'CJS370H1',
 'CLT334H1',
 'CLT335H1',
 'COG100H1',
 'CRE283H1',
 'CRE367H1',
 'CRI387H1',
 'CSC099Y1',
 'CSC374H1',
 'CSC394H1',
 'CSC395H1',
 'CSE260H1',
 'DTS398H0',
 'EAS214H1',
 'EAS225H1',
 'EAS316H1',
 'EAS415H1',
 'EAS422H1',
 'ECO099Y1',
 'ECO357H1',
 'ECO395H1',
 'ECO452H1',
 'ECO484H1',
 'EEB403H0',
 'EEB405H0',
 'EEB406H0',
 'ENG366H1',
 'ENG395H1',
 'ENG496H1',
 'ENT310H1',
 'ESS234H0',
 'ESS324H0',
 'ESS410H0',
 'ESS450H0',
 'EST208H1',
 'FAH275H1',
 'FAH305H1',
 'FAH387H1',
 'FAH389H1',
 'FCS185H1',
 'FCS186H1',
 'FCS298H1',
 'FLC099Y1',
 'FSL100H0',
 'FSL102H0',
 'FSL110Y1',
 'FSL322H0',
 'FSL420H0',
 'FSL442H0',
 'FSL443H0',
 'GER230H1',
 'HIS271H1',
 'HIS272H1',
 'HIS357H1',
 'HIS389Y0',